<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

## Retrieval-Augmented Generation with SAP AI Core Document Grounding

In this demo, we will explore how to enhance the capabilities of Large Language Models (LLMs) with **SAP AI Core Document Grounding**. You will learn how to index unstructured and semi-structured data using AI models from **SAP Generative AI Hub**, and store the document chunks in **SAP AI Core Document Grounding**. Additionally, you will retrieve relevant document chunks using semantic search, and forward the relevant results to a LLM to generate an augmented answer.

## 🎯 Learning Objectives
By the end of this demo, you will be able to:
- Implement a full RAG pipeline using Python, LangChain, and Generative AI Hub SDK.
- Index document chunks into SAP AI Core Document Grounding collections.
- Retrieve the most relevant content based on semantic similarity using Document Grounding retrieval.
- Augment user prompts with retrieved context and invoke LLMs to generate more accurate, grounded answers.
- Design and use prompt templates to enhance the quality of generated responses.

## 🚨 Requirements

Before starting the Jupyter Notebook steps, ensure the following:
- SAP AI Core instance with **Document Grounding** capability enabled
- Deploy AI models in SAP AI Launchpad:
  - Large Language Model (LLM) for chat/completion: **`gpt-4.1-mini`**
  - Embedding model for vector representations: **`text-embedding-3-large`**

## 📝 About the Data

The data set is a product catalog of IT accessory products. Here are the main attributes and their descriptions based on the sample data:

|Field          |Description            |
----------------|-----------------------
|**PRODUCT_ID**| A unique identifier for each product.|
|**PRODUCT_NAME**| The name of the product, which typically includes the brand and the model.|
|**CATEGORY**| The general category of the product, which is "IT Accessories" for all entries sampled.|
|**DESCRIPTION**| A detailed description of the product, highlighting key features and specifications.|
|**UNIT_PRICE**| The price of the product in Euros.|
|**UNIT_MEASURE**| The unit of measure for the product, typically "Each" indicating pricing per item.|
|**SUPPLIER_ID**| A unique identifier for the supplier of the product.|
|**SUPPLIER_NAME**| The name of the supplier.|
|**LEAD_TIME_DAYS**| The number of days it takes from order to delivery.|
|**MIN_ORDER**| The minimum order quantity required.|
|**CURRENCY**| The currency of the transaction, which is "EURO" for all entries.|
|**SUPPLIER_COUNTRY**| The country where the supplier is located, which is "Germany" for all sampled entries.|
|**SUPPLIER_ADDRESS**| The physical address of the supplier.|
|**AVAILABILITY_DAYS**| The number of days the product is available for delivery.|
|**SUPPLIER_CITY**| The city where the supplier is located.|
|**STOCK_QUANTITY**| The quantity of the product currently in stock.|
|**MANUFACTURER**| The company that manufactured the product.|
|**CITY_LAT**| Geographical coordinates of the city (latitude)|
|**CITY_LONG**| Geographical coordinates of the city (longitude).|
|**RATING**| A rating for the product, which are on a scale from 1 to 5.|

</br>

This dataset is structured to support various business operations such as inventory management, order processing, and logistics planning, providing a comprehensive view of product offerings, supplier details, and stock levels.

</br>


## Retrieval Augmented Generation using Generative AI Hub and SAP AI Core Document Grounding

### Hands-on Retrieval Augmented Generation (RAG) workflow

The Retrieval Augmented Generation use case process consists of steps to be completed as seen in the graphic below.

<br>

> ![title](./images/rag_full.png)

<br>

#### Indexing Process
1. Business documents that should be used for answering user questions are fed into the model. The contents of the files are split into smaller chunks.
    >"Chunking" refers to dividing a large text corpus into smaller, manageable pieces or segments. Each chunk acts as a standalone unit of information that can be individually indexed and retrieved.
2. Embedding functions are used to create embeddings from the file/document chunks.
    >Embeddings refer to dense, continuous vectors representing text in a high-dimensional space. These vectors serve as coordinates in a semantic space, capturing the relationships and meanings between words.
3. The embeddings are then stored as vectors in the **SAP AI Core Document Grounding** service.

#### Retrieval Process
1. A query or prompt is submitted.
2. The query is embedded into a vector form.
3. The query vector is compared to the values stored in SAP AI Core Document Grounding via a similarity/semantic search.
4. The most appropriate and relevant results are identified.
5. And forwarded, along with the original query, to a large language model.
6. The LLM uses the results of the Document Grounding search to augment its own searching capabilities, and the final answer is returned to the user.

### Implementing RAG with Document Grounding

- Prepare the documentation for the product catalog in CSV format with each row representing a product.
- Connect to the SAP AI Core Document Grounding service and create a **collection** to store the documentation data.
- Populate the collection with document chunks using the Document Grounding REST API.
- Use the Document Grounding retrieval API to perform semantic search and retrieve relevant results.

### Enhancing Query Responses

- Define a prompt template to provide context to queries.
- Modify the function to query the LLM (Large Language Model) based on the prompt template.
- Test the model's response using specific queries related to the product catalog, ensuring it provides contextually relevant responses based on retrieved document chunks.

> Retrieval augmented generation optimizes the output of large language models by applying more context to prompts.

## Setup and configuration

The following Python modules are to be installed during this hands-on introduction.

#### **sap-ai-sdk-gen**

With this SAP Python SDK you can leverage the power of generative models available in SAP Generative AI Hub. It provides:
- `AICoreV2Client` and `GenAIHubProxyClient` for authentication
- `VectorAPIClient` for Document Grounding collection management and semantic search
- `OrchestrationService` with `GroundingModule` for end-to-end RAG in a single API call

For more information, please see https://pypi.org/project/sap-ai-sdk-gen/


#### Install Python packages

Run the following package installation. **pip** is the package installer for Python.

In [ ]:
%pip install "sap-ai-sdk-gen[all]" --break-system-packages -U

# kernel restart required!!!

#### Restart Python kernel

The Python kernel needs to be restarted before continuing.

> ![title](./images/config_001.png)

#### Configure SAP Generative AI Hub credentials

Execute the configuration module below to enable access to SAP Generative AI foundation models. This configuration is automatically done by running configuration module in the code block.

You could also set up the same by running a terminal command: **aicore configure**

</br>

> Please ensure that the Python kernel was restarted!


In [ ]:
# Generative AI Config
import os
import json

with open(os.path.join(os.getcwd(), 'env_config.json')) as f:
    aicore_config = json.load(f)

### Initialize the AI Core Client and Orchestration Service

We set up the `AICoreV2Client` and `GenAIHubProxyClient` for authentication, and initialize the `OrchestrationService` which will be used for both the "without context" demo and the full RAG pipeline.

> **IMPORTANT!** The Orchestration Service is deployed by default with SAP Generative AI Hub — no additional deployment is needed.

In [ ]:
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from gen_ai_hub.orchestration_v2 import OrchestrationService

# Set up the AICoreV2Client
ai_core_client = AICoreV2Client(
    base_url=aicore_config['AICORE_BASE_URL'],
    auth_url=aicore_config['AICORE_AUTH_URL'],
    client_id=aicore_config['AICORE_CLIENT_ID'],
    client_secret=aicore_config['AICORE_CLIENT_SECRET'],
    resource_group=aicore_config['AICORE_RESOURCE_GROUP']
)
# Set up the GenAIHubProxyClient
proxy_client = GenAIHubProxyClient(ai_core_client=ai_core_client)

# Initialize the Orchestration Service — discover the running deployment dynamically
orchestration_deployment = next(
    d for d in ai_core_client.deployment.query().resources
    if 'orchestration' in d.scenario_id.lower() and d.status.value == 'RUNNING'
)
orchestration_url = f"{aicore_config['AICORE_BASE_URL']}/inference/deployments/{orchestration_deployment.id}"
orchestration_service = OrchestrationService(api_url=orchestration_url, proxy_client=proxy_client)

print("✅ AI Core Client connection is established successfully!")
print("✅ Orchestration Service (V2) initialized!")
print(f"   Deployment ID : {orchestration_deployment.id}")

### Ask LLM without context

After completing the configuration we are ready to ask the first question directly to the LLM without any business product context to find us products with a rating of 4 and more. The response is arbitrary and does not relate to our product data.

</br>

> **Note** We can solve this problem by following the next steps in implementing RAG with Document Grounding.

In [ ]:
from IPython.display import Markdown
from gen_ai_hub.orchestration_v2 import (
    OrchestrationConfig, ModuleConfig, LLMModelDetails,
    PromptTemplatingModuleConfig, Template, SystemMessage, UserMessage
)

question = "Return one keyboard suggestion (brand + model) with rating greater than 4. Format as: 'Keyboard: <name>'"

# --- Configuration ---
LLM_MODEL = "gpt-4.1-mini"
LLM_TEMPERATURE = 0.3
LLM_MAX_TOKENS = 800

# Call the LLM directly via Orchestration Service V2 — no grounding, no context
no_context_config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=PromptTemplatingModuleConfig(
            prompt=Template(template=[
                SystemMessage(content="Advise the user based on the information available to you from general sources."),
                UserMessage(content="{{?question}}")
            ]),
            model=LLMModelDetails(name=LLM_MODEL, params={"max_tokens": LLM_MAX_TOKENS, "temperature": LLM_TEMPERATURE})
        )
    )
)

response = orchestration_service.run(
    config=no_context_config,
    placeholder_values={"question": question}
)

display(Markdown(response.final_result.choices[0].message.content.strip()))

# Implementing Retrieval Augmented Generation (RAG)

### Prepare the documentation for the product catalog in CSV format with each row representing a product

This code snippet demonstrates how to load and process text data from a CSV file using the `CSVLoader` from the `langchain_community.document_loaders` library.

This process is useful for handling large text data, making it more manageable or suitable for further processing, analysis, or input into machine learning models, especially when dealing with limitations on input size.

In [2]:
import csv

def load_csv_documents(file_path: str) -> list:
    """Load CSV rows as simple document objects (replaces LangChain CSVLoader)."""
    content_columns = [
        "PRODUCT_NAME", "DESCRIPTION", "UNIT_PRICE", "LEAD_TIME_DAYS",
        "STOCK_QUANTITY", "RATING", "MIN_ORDER", "CATEGORY", "SUPPLIER_NAME",
        "SUPPLIER_COUNTRY", "SUPPLIER_CITY", "SUPPLIER_ADDRESS", "STATUS",
        "CURRENCY", "MANUFACTURER", "CITY_LAT", "CITY_LONG",
    ]
    metadata_columns = ["SUPPLIER_ID", "CATEGORY", "SUPPLIER_COUNTRY", "SUPPLIER_CITY", "MANUFACTURER"]

    class Document:
        def __init__(self, page_content, metadata):
            self.page_content = page_content
            self.metadata = metadata

    docs = []
    with open(file_path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";", quotechar='"')
        for row in reader:
            content = "\n".join(
                f"{col}: {row[col]}" for col in content_columns if col in row
            )
            metadata = {col: row[col] for col in metadata_columns if col in row}
            metadata["source"] = row.get("PRODUCT_ID", "")
            docs.append(Document(page_content=content, metadata=metadata))
    return docs


> At this point we have implemented the first RAG step - generated text chunks from source data
> 
> ![rag_indexing_1](./images/rag_indexing_1.png)

### SAP AI Core Document Grounding

[The **grounding module** in SAP AI Core](https://help.sap.com/docs/sap-ai-core/generative-ai/grounding-035c455a5a424697b60f4a24b6d791fe) integrates external, domain-specific, or real-time data into generative AI workflows. By enriching pretrained models with contextually relevant information, grounding improves response accuracy and relevance beyond the models' general training data.

Pretrained generative models are trained on broad, general-purpose datasets and do not have direct access to your organization's proprietary or real-time data. Grounding addresses this limitation by:
- Connecting models to external data sources
- Retrieving relevant information at runtime
- Supplying that information as context for generation

#### Two Approaches to Prepare the Knowledge Base

SAP AI Core Document Grounding offers two options for providing data:

| | Option 1: Pipelines API | Option 2: Vector API (this notebook) |
|---|---|---|
| **How it works** | Upload documents to a supported data repository (SharePoint, S3, SFTP, SAP DMS, etc.) and run a data pipeline that automatically chunks, embeds, and indexes the documents | Provide document chunks directly — you control chunking and content |
| **Best for** | Large document repositories with scheduled refresh (daily sync) | Structured data, custom chunking logic, or programmatic ingestion |
| **Supported sources** | Microsoft SharePoint, AWS S3, SFTP, SAP Build Work Zone, SAP Document Management, ServiceNow, Google Drive | Any data you can load in Python |
| **SDK client** | `PipelineAPIClient` | `VectorAPIClient` |

<br/>

> **This notebook uses Option 2 (Vector API).** We load the product catalog CSV in Python, convert each row into a document chunk, and upload it to a Document Grounding collection using the `VectorAPIClient` SDK. This gives us full control over the content and metadata of each chunk.

#### Grounding Architecture

![image.png](images/ai_core_grounding_architecture.png)

</br>

#### Connect to the SAP AI Core Document Grounding service

We initialize the `VectorAPIClient` from the SAP Generative AI Hub SDK. It reuses the `proxy_client` already set up above — no separate OAuth2 token management needed.

In [ ]:
from gen_ai_hub.document_grounding import VectorAPIClient

# Initialize the VectorAPIClient using the existing proxy_client
vector_client = VectorAPIClient(proxy_client=proxy_client)

print("✅ Document Grounding VectorAPIClient initialized!")

## Create a Document Grounding Collection and Store Embeddings

We will create a **collection** in SAP AI Core Document Grounding to store our product catalog documents. A collection is a logical container that groups related documents and their vector embeddings together.

The collection is configured with an embedding model (`text-embedding-3-large`) that will be used to automatically generate vector representations of the uploaded document chunks.

### Create a collection in Document Grounding

A collection in Document Grounding is equivalent to a table in a vector database. It stores document chunks along with their vector embeddings and metadata, enabling fast semantic search.

> **Note:** Collection creation is **asynchronous**. The API returns `202 Accepted` with a `Location` header pointing to a status URL. We poll that URL until the collection status is `created`.

In [ ]:
import time
from gen_ai_hub.document_grounding import CollectionCreateRequest, EmbeddingConfig

collection_title = "products-it-accessories"

# Snapshot existing collection IDs before creation
before_ids = {c.id for c in vector_client.get_collections().resources}

try:
    create_response = vector_client.create_collection(
        CollectionCreateRequest(
            title=collection_title,
            embeddingConfig=EmbeddingConfig(modelName="text-embedding-3-large")
        )
    )
except Exception as e:
    if "Expecting value" in str(e):
        # 202 empty body — creation accepted, find the NEW collection (not in before_ids)
        time.sleep(5)
        after = vector_client.get_collections().resources
        new_cols = [c for c in after if c.id not in before_ids and c.title == collection_title]
        if not new_cols:
            raise RuntimeError(f"New collection '{collection_title}' not found after creation") from e
        collection_id = new_cols[0].id
        create_response = None
    else:
        raise
else:
    location = create_response.headers.get("Location", "")
    collection_id = location.split("/collections/")[-1].split("/")[0]

print(f"   Collection ID: {collection_id}")

# Poll until ready
print(f"   Waiting for collection '{collection_title}' to be ready...")
for attempt in range(30):
    status_resp = vector_client.get_collection_creation_status(collection_id)
    current_status = status_resp.status if hasattr(status_resp, 'status') else 'CREATED'
    if current_status == 'CREATED':
        print(f"✅ Collection created successfully!")
        print(f"   Collection ID   : {collection_id}")
        print(f"   Collection title: {collection_title}")
        break
    print(f"   Attempt {attempt + 1}: status={current_status}, retrying...")
    time.sleep(2)
else:
    raise TimeoutError("Collection creation timed out after 60 seconds")


### Upload product documents to the collection

We convert the loaded CSV documents into the Document Grounding format and upload them to the collection. Each document consists of:
- **metadata**: key-value pairs where each value is a **list of strings** (required by the API schema)
- **chunks**: the actual text content that will be embedded and indexed for retrieval

In [ ]:
from gen_ai_hub.document_grounding import (
    DocumentsCreateRequest, BaseDocument, TextOnlyBaseChunk, VectorKeyValueListPair
)

BATCH_SIZE = 10  # Number of documents per upload batch

def upload_documents_batch(documents_batch):
    """Upload a batch of documents to the Document Grounding collection."""
    vector_client.create_documents(
        collection_id,
        DocumentsCreateRequest(
            documents=[
                BaseDocument(
                    metadata=[
                        VectorKeyValueListPair(key=k, value=[str(v)])
                        for k, v in doc.metadata.items() if v is not None
                    ],
                    chunks=[TextOnlyBaseChunk(content=doc.page_content, metadata=[])]
                )
                for doc in documents_batch
            ]
        )
    )

# Delete all existing documents before upload (ensures clean state on re-run)
existing = vector_client.get_documents(collection_id, top=1)
if len(existing.resources) > 0:
    print(f"🗑️  Deleting existing documents from collection (clean re-index)...")
    all_docs = vector_client.get_documents(collection_id, top=1000)
    for doc in all_docs.resources:
        try:
            vector_client.delete_document(collection_id, doc.id)
        except Exception as e:
            if "Expecting value" not in str(e):
                raise  # only swallow the empty-body 204 parse error
    print(f"   Deleted {len(all_docs.resources)} documents")

# Upload all documents in batches
total_uploaded = 0
for i in range(0, len(text_documents), BATCH_SIZE):
    batch = text_documents[i : i + BATCH_SIZE]
    upload_documents_batch(batch)
    total_uploaded += len(batch)
    print(f"   Uploaded batch {i // BATCH_SIZE + 1}: {total_uploaded}/{len(text_documents)} documents")
print(f"✅ All {total_uploaded} product documents uploaded to collection '{collection_title}'!")


### Verify documents stored in the collection

After uploading, we verify that the documents have been successfully stored in the Document Grounding collection by retrieving the **top 5 documents** and inspecting their metadata.

In [ ]:
# Retrieve the top 5 documents stored in the collection
# get_documents returns DocumentsResponse: { count, resources: List[DocumentWithoutChunks] }
stored_docs = vector_client.get_documents(collection_id, top=5)

print(f"✅ Documents in collection '{collection_title}':")
print(f"   Number of documents retrieved: {stored_docs.count}")
print()

for i, doc in enumerate(stored_docs.resources[:3]):
    # DocumentWithoutChunks: { id, metadata: List[VectorKeyValueListPair] }
    print(f"--- Document {i + 1} ---")
    print(f"ID: {doc.id}")
    for m in doc.metadata[:3]:
        print(f"  {m.key}: {m.value}")
    print()

> At this point we have implemented the Indexing Process part of RAG
> 
> ![rag_indexing_2](./images/rag_indexing_2.png)

## Performing a Semantic Search to Find Relevant Products

## Vector Search Using SAP AI Core Document Grounding

In this step, we use the **`VectorAPIClient.search()`** method to convert a natural language query into a vector representation and retrieve the most relevant product records from the collection using **semantic similarity search**.

The search request (`TextSearchRequest`) accepts:
- A natural language **`query`** string
- A list of **`filters`** (`VectorSearchFilter`) specifying which collections to search and how many results to return

The SDK automatically embeds the query and returns the most semantically similar document chunks.

In [ ]:
from gen_ai_hub.document_grounding import (
    TextSearchRequest, VectorSearchFilter, VectorSearchConfiguration
)

def search_documents(query, collection_id, top_k=4):
    """Search for relevant documents using semantic similarity via VectorAPIClient."""
    # VectorSearchResults: { results: List[VectorPerFilterSearchResult] }
    return vector_client.search(
        TextSearchRequest(
            query=query,
            filters=[
                VectorSearchFilter(
                    id="filter-1",
                    collectionIds=[collection_id],
                    configuration=VectorSearchConfiguration(maxDocumentCount=top_k)
                )
            ]
        )
    )

def extract_chunks(search_results):
    """Extract chunk content strings from VectorSearchResults.
    VectorSearchResults.results -> VectorPerFilterSearchResult
      .results -> DocumentsChunk
        .documents -> DocumentOutput
          .chunks -> VectorChunk { id, content, metadata }
    """
    chunks = []
    for filter_result in search_results.results:       # VectorPerFilterSearchResult
        for collection_result in filter_result.results:  # DocumentsChunk
            for doc in collection_result.documents:       # DocumentOutput
                for chunk in doc.chunks:                  # VectorChunk
                    chunks.append(chunk.content)
    return chunks

# Test the semantic search
search_query = "keyboard with rating 4 or higher"
search_results = search_documents(search_query, collection_id, top_k=4)
chunks = extract_chunks(search_results)

print(f"Search query: '{search_query}'")
print(f"Retrieved {len(chunks)} chunk(s)")
print()

for i, content in enumerate(chunks):
    print(f"--- Chunk {i + 1} ---")
    print(content)
    print()

> At this point we have implemented the Retrieval Process part of RAG
> 
> ![rag_retrieval_1](./images/rag_retrieval_1.png)

## Augmenting the LLM with Retrieved Context

Now we combine the retrieved document chunks with the user's question and pass them to the LLM. This is the **Augmented Generation** step of RAG.

We use the **SAP Generative AI Hub Orchestration Service**, which provides a higher-level abstraction that handles both retrieval and generation in a single API call. The `GroundingModule` automatically:
1. Embeds the user question
2. Searches the Document Grounding collection for relevant chunks
3. Injects the retrieved context into the prompt
4. Calls the LLM with the augmented prompt


### Configure the Grounding Module

We configure the `GroundingModule` to search our product catalog collection. The `DocumentGroundingFilter` specifies:
- **`data_repositories`**: the collection ID created in the indexing step
- **`data_repository_type`**: `VECTOR` (our collection uses the Vector API)
- **`max_chunk_count`**: how many chunks to retrieve per query

In [ ]:
from gen_ai_hub.orchestration_v2 import (
    OrchestrationConfig, ModuleConfig, LLMModelDetails,
    PromptTemplatingModuleConfig, Template, SystemMessage, UserMessage,
    GroundingModuleConfig, GroundingType, DocumentGroundingConfig,
    DocumentGroundingPlaceholders, DocumentGroundingFilter,
    DataRepositoryType, GroundingSearchConfig
)

# --- Configuration ---
LLM_MODEL = "gpt-4.1-mini"
LLM_TEMPERATURE = 0.3
LLM_MAX_TOKENS = 800
MAX_CHUNK_COUNT = 4  # Number of product catalog chunks to retrieve per query

# Full orchestration config: template + LLM + grounding (V2)
rag_config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=PromptTemplatingModuleConfig(
            prompt=Template(template=[
                SystemMessage(content=(
                    "You are a helpful product advisor. "
                    "Use ONLY the following product catalog context to answer the question. "
                    "If the answer is not found in the context, say 'I could not find relevant products in the catalog.'"
                )),
                UserMessage(content=(
                    "Context: {{?grounding_response}}\n\n"
                    "Question: {{?question}}\n\n"
                    "Answer based on the catalog data above:"
                ))
            ]),
            model=LLMModelDetails(
                name=LLM_MODEL,
                params={"temperature": LLM_TEMPERATURE, "max_tokens": LLM_MAX_TOKENS}
            )
        ),
        grounding=GroundingModuleConfig(
            type=GroundingType.DOCUMENT_GROUNDING_SERVICE,
            config=DocumentGroundingConfig(
                filters=[DocumentGroundingFilter(
                    id="filter-1",
                    data_repositories=[collection_id],
                    search_config=GroundingSearchConfig(max_chunk_count=MAX_CHUNK_COUNT),
                    data_repository_type=DataRepositoryType.VECTOR
                )],
                placeholders=DocumentGroundingPlaceholders(
                    input=["question"],
                    output="grounding_response"
                )
            )
        )
    )
)

print("✅ RAG configuration ready!")
print(f"   Collection ID : {collection_id}")
print(f"   LLM model     : {LLM_MODEL}")
print(f"   Max chunks    : {MAX_CHUNK_COUNT}")

### Test the RAG pipeline with product catalog queries

Now we test the complete RAG pipeline with questions about our product catalog. The Orchestration Service handles retrieval and generation in a single call.

In [ ]:
# Test Query 1: Find keyboards with high ratings
question1 = "Return one keyboard suggestion (brand + model) with rating >=4. Format as: 'Keyboard: <name>'"
print(f"Question: {question1}")
print()

response1 = orchestration_service.run(
    config=rag_config,
    placeholder_values={"question": question1}
)

display(Markdown(response1.final_result.choices[0].message.content.strip()))


In [ ]:
# Test Query 2: Find products from a specific supplier city
question2 = "Which products are supplied from Germany? List up to 3 with their prices."
print(f"Question: {question2}")
print()

response2 = orchestration_service.run(
    config=rag_config,
    placeholder_values={"question": question2}
)
display(Markdown(response2.final_result.choices[0].message.content.strip()))

In [ ]:
# Test Query 3: Find products under a price threshold
question3 = "What are the cheapest mice available? List up to 3 with their prices."
print(f"Question: {question3}")
print()

response3 = orchestration_service.run(
    config=rag_config,
    placeholder_values={"question": question3}
)
display(Markdown(response3.final_result.choices[0].message.content.strip()))

## Summary

In this notebook, we successfully implemented a complete **Retrieval-Augmented Generation (RAG)** pipeline using **SAP AI Core Document Grounding** and the **Orchestration Service**:

1. **Data Loading**: Loaded the product catalog from a CSV file using LangChain's `CSVLoader`.
2. **Connection**: Initialized `VectorAPIClient` and `OrchestrationService` using the SAP Generative AI Hub SDK — no manual OAuth2 token management needed.
3. **Indexing**: Created a collection using `VectorAPIClient.create_collection()` and uploaded all product documents via `VectorAPIClient.create_documents()`.
4. **Retrieval**: Used `VectorAPIClient.search()` with a `TextSearchRequest` and `VectorSearchFilter` to find relevant product chunks for user queries.
5. **Augmented Generation**: Used the **Orchestration Service** with `GroundingModule` to automatically retrieve context and generate grounded answers in a single API call.

### Orchestration Service vs Manual RAG

| | Manual RAG (VectorAPIClient + direct search) | Orchestration Service |
|---|---|---|
| **Search** | `VectorAPIClient.search()` (manual) | Handled automatically by `GroundingModule` |
| **Prompt building** | Manual template + context injection | Declarative `Template` with `{{ ?grounding_response }}` |
| **LLM call** | `OrchestrationService.run()` without grounding | Handled by `OrchestrationConfig` with grounding |
| **Code complexity** | Higher | Lower |
| **Production readiness** | Good | Better |

### Key Benefits of SAP AI Core Document Grounding
- **Fully managed**: No need to deploy or maintain a separate vector database
- **Integrated**: Seamlessly works with SAP Generative AI Hub models and Orchestration Service via the native SDK
- **Scalable**: Handles enterprise-scale document collections
- **Secure**: Authentication handled transparently by `GenAIHubProxyClient`